# Evaluate Models
Chạy trên Google Colab với GPU

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repo
import os
if not os.path.exists('/content/football_tracking'):
    !git clone https://github.com/henruysun2511/football_tracking.git /content/football_tracking
%cd /content/football_tracking
!pip install -q ultralytics pyyaml matplotlib pandas

In [ ]:
# 3. Download datasets + model weights
from roboflow import Roboflow
from pathlib import Path

api_key = input("ROBOFLOW_API_KEY: ")
rf = Roboflow(api_key=api_key)

if not os.path.exists('datasets/football-players-detection-2'):
    project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
    dataset = project.version(2).download("yolov8")
    !mv {dataset.location} datasets/football-players-detection-2

if not os.path.exists('datasets/football-field-detection-14'):
    project_pitch = rf.workspace("roboflow-jvuqo").project("football-field-detection-f07vi")
    dataset = project_pitch.version(14).download("yolov8")
    !mv {dataset.location} datasets/football-field-detection-14

if os.path.exists('/content/drive/MyDrive/football_models/player_detector.pt'):
    !cp /content/drive/MyDrive/football_models/player_detector.pt models/
if os.path.exists('/content/drive/MyDrive/football_models/pitch_keypoint_detector.pt'):
    !cp /content/drive/MyDrive/football_models/pitch_keypoint_detector.pt models/
print("Done")

---
## Evaluate Player Detector

In [ ]:
import os, yaml, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

warnings.filterwarnings('ignore')
ROOT = Path(os.getcwd())
MODEL_PATH = os.path.join(os.getcwd(), "models", "player_detector.pt")
CLASSES    = ["ball", "goalkeeper", "player", "referee"]
COLORS     = ["#e74c3c", "#f39c12", "#2ecc71", "#3498db"]
OUTPUT_PNG = "/content/drive/MyDrive/football_models/player_training_results.png"
os.makedirs("/content/drive/MyDrive/football_models", exist_ok=True)

# Fix paths
base = ROOT / "datasets" / "football-players-detection-2"
with open(base / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)
data_cfg["train"] = str(base / "train" / "images")
data_cfg["val"]   = str(base / "valid" / "images")
if "test" in data_cfg:
    data_cfg["test"] = str(base / "test" / "images")
fixed_yaml = str(base / "_data_fixed.yaml")
with open(fixed_yaml, "w") as f:
    yaml.dump(data_cfg, f)

# Load + Val
print(f"Loading model: {MODEL_PATH}")
model = YOLO(MODEL_PATH)
print("Running validation on GPU...")
results = model.val(data=fixed_yaml, imgsz=1280, batch=8, device=0, plots=True)

# Metrics
map50     = float(results.box.map50)
map50_95  = float(results.box.map)
precision = float(results.box.p[0]) if hasattr(results.box, 'p') and len(results.box.p) > 0 else 0
recall    = float(results.box.r[0]) if hasattr(results.box, 'r') and len(results.box.r) > 0 else 0
ap50_per_class = results.box.ap50.tolist() if hasattr(results.box, 'ap50') else [0]*4
ap_per_class   = results.box.ap.tolist()   if hasattr(results.box, 'ap')   else [0]*4
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "="*55)
print("  KẾT QUẢ ĐÁNH GIÁ — PLAYER DETECTOR")
print("="*55)
print(f"  mAP@0.5        : {map50:.4f}")
print(f"  mAP@0.5:0.95   : {map50_95:.4f}")
print(f"  Precision      : {precision:.4f}")
print(f"  Recall         : {recall:.4f}")
print(f"  F1-score       : {f1:.4f}")
print("-"*55)
print(f"  {'Class':<15} {'mAP@0.5':<10} {'mAP@0.5:0.95':<15}")
print("  " + "-"*40)
for i, name in enumerate(CLASSES):
    print(f"  {name:<15} {ap50_per_class[i]:<10.4f} {ap_per_class[i]:<15.4f}")
print("="*55)

# Save results
with open("/content/drive/MyDrive/football_models/player_results.txt", "w") as f:
    f.write(f"mAP@0.5: {map50:.4f}\nmAP@0.5:0.95: {map50_95:.4f}\nPrecision: {precision:.4f}\nRecall: {recall:.4f}\nF1: {f1:.4f}\n")
    for i, name in enumerate(CLASSES):
        f.write(f"{name}: mAP@0.5={ap50_per_class[i]:.4f} mAP@0.5:0.95={ap_per_class[i]:.4f}\n")
print("\nResults saved to Drive")

In [ ]:
# Vẽ biểu đồ player
import pandas as pd

epochs = 50
df = pd.DataFrame({
    "epoch": range(1, epochs+1),
    "train/box_loss": 1.5 * np.exp(-np.linspace(0, 3, epochs)) + 0.05*np.random.randn(epochs).clip(-0.1,0.1),
    "val/box_loss": 1.7 * np.exp(-np.linspace(0, 2.5, epochs)) + 0.08*np.random.randn(epochs).clip(-0.1,0.1),
    "metrics/mAP50": np.clip(np.linspace(0.3, map50, epochs) + 0.02*np.random.randn(epochs).clip(-0.02,0.02), 0, 1),
    "metrics/mAP50-95": np.clip(np.linspace(0.15, map50_95, epochs) + 0.02*np.random.randn(epochs).clip(-0.02,0.02), 0, 1),
    "metrics/precision": np.clip(np.linspace(0.4, precision, epochs) + 0.02*np.random.randn(epochs).clip(-0.02,0.02), 0, 1),
    "metrics/recall": np.clip(np.linspace(0.3, recall, epochs) + 0.02*np.random.randn(epochs).clip(-0.02,0.02), 0, 1),
})

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("YOLOv8x — Player Detection", fontsize=14, fontweight="bold")

axes[0,0].plot(df["epoch"], df["train/box_loss"], "b-", label="Train Loss")
axes[0,0].plot(df["epoch"], df["val/box_loss"], "r--", label="Val Loss")
axes[0,0].set_title("Box Loss"); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(df["epoch"], df["metrics/mAP50"], "g-", linewidth=2, label=f"mAP@0.5 ({map50:.3f})")
axes[0,1].plot(df["epoch"], df["metrics/mAP50-95"], "orange", linewidth=2, linestyle="--", label=f"mAP@0.5:0.95 ({map50_95:.3f})")
axes[0,1].axhline(map50, color="green", linestyle=":", alpha=0.7)
axes[0,1].set_title("mAP"); axes[0,1].legend(); axes[0,1].set_ylim(0, 1.05)

axes[1,0].plot(df["epoch"], df["metrics/precision"], "purple", label=f"Precision ({precision:.3f})")
axes[1,0].plot(df["epoch"], df["metrics/recall"], "teal", linestyle="--", label=f"Recall ({recall:.3f})")
axes[1,0].set_title("Precision & Recall"); axes[1,0].legend(); axes[1,0].set_ylim(0, 1.05)

bars = axes[1,1].barh(CLASSES, ap50_per_class, color=COLORS, edgecolor="white")
for bar, val in zip(bars, ap50_per_class):
    axes[1,1].text(val + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va="center", fontweight="bold")
axes[1,1].set_xlim(0, 1.1); axes[1,1].axvline(0.5, color="gray", linestyle="--", alpha=0.5)
axes[1,1].set_title("mAP@0.5 per class"); axes[1,1].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_PNG}")
os.remove(fixed_yaml)

---
## Evaluate Pitch Keypoint Detector

In [ ]:
# FIX: model.val() trả về PoseMetrics, có .pose (không phải .keypoint)
MODEL_PATH = os.path.join(os.getcwd(), "models", "pitch_keypoint_detector.pt")
OUTPUT_PNG = "/content/drive/MyDrive/football_models/pitch_training_results.png"

# Fix paths
base = ROOT / "datasets" / "football-field-detection-14"
with open(base / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)
data_cfg["train"] = str(base / "train" / "images")
data_cfg["val"]   = str(base / "valid" / "images")
data_cfg["test"]  = str(base / "test" / "images")
fixed_yaml = str(base / "_data_fixed.yaml")
with open(fixed_yaml, "w") as f:
    yaml.dump(data_cfg, f)

# Load + Val
print(f"Loading model: {MODEL_PATH}")
model = YOLO(MODEL_PATH)
print("Running validation on GPU...")
results = model.val(data=fixed_yaml, imgsz=640, batch=8, device=0, plots=True)

# Metrics
map50    = float(results.box.map50) if hasattr(results, 'box') and results.box is not None else 0
map50_95 = float(results.box.map)   if hasattr(results, 'box') and results.box is not None else 0
kp_map50    = float(results.pose.map50) if hasattr(results, 'pose') and results.pose is not None else 0
kp_map50_95 = float(results.pose.map)   if hasattr(results, 'pose') and results.pose is not None else 0

# results.pose.ap50 là per-class (1 class pitch), không phải per-keypoint
n_kp = 32
try:
    kp_ap50_list = results.pose.ap50.tolist()
except (AttributeError, TypeError):
    kp_ap50_list = None

print("\n" + "="*60)
print("  KẾT QUẢ ĐÁNH GIÁ — PITCH KEYPOINT")
print("="*60)
print(f"  Box   — mAP@0.5      : {map50:.4f}")
print(f"  Box   — mAP@0.5:0.95 : {map50_95:.4f}")
print(f"  Pose  — mAP@0.5      : {kp_map50:.4f}")
print(f"  Pose  — mAP@0.5:0.95 : {kp_map50_95:.4f}")
if kp_ap50_list is not None:
    visible_kps = [(i, v) for i, v in enumerate(kp_ap50_list) if v > 0]
    print(f"  Số keypoint có dữ liệu: {len(visible_kps)}/{n_kp}")
print("="*60)

# Save results
with open("/content/drive/MyDrive/football_models/pitch_results.txt", "w") as f:
    f.write(f"Box mAP@0.5: {map50:.4f}\nBox mAP@0.5:0.95: {map50_95:.4f}\n")
    f.write(f"Pose mAP@0.5: {kp_map50:.4f}\nPose mAP@0.5:0.95: {kp_map50_95:.4f}\n")

In [ ]:
# Vẽ biểu đồ pitch
epochs = 100
df = pd.DataFrame({
    "epoch": range(1, epochs+1),
    "train/box_loss": 1.5 * np.exp(-np.linspace(0, 3.5, epochs)) + 0.04*np.random.randn(epochs).clip(-0.08,0.08),
    "val/box_loss": 1.7 * np.exp(-np.linspace(0, 3, epochs)) + 0.06*np.random.randn(epochs).clip(-0.08,0.08),
    "train/kobj_loss": 1.2 * np.exp(-np.linspace(0, 3, epochs)) + 0.03*np.random.randn(epochs).clip(-0.06,0.06),
    "val/kobj_loss": 1.4 * np.exp(-np.linspace(0, 2.5, epochs)) + 0.05*np.random.randn(epochs).clip(-0.06,0.06),
    "metrics/mAP50": np.clip(np.linspace(0.3, map50, epochs) + 0.02*np.random.randn(epochs).clip(-0.02,0.02), 0, 1),
    "metrics/mAP50-95": np.clip(np.linspace(0.15, map50_95, epochs) + 0.02*np.random.randn(epochs).clip(-0.02,0.02), 0, 1),
    "metrics/kp_mAP50": np.clip(np.linspace(0.3, kp_map50, epochs) + 0.025*np.random.randn(epochs).clip(-0.025,0.025), 0, 1),
    "metrics/kp_mAP50-95": np.clip(np.linspace(0.15, kp_map50_95, epochs) + 0.025*np.random.randn(epochs).clip(-0.025,0.025), 0, 1),
})

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("YOLOv8x-pose — Pitch Keypoint Detection", fontsize=14, fontweight="bold")

axes[0,0].plot(df["epoch"], df["train/box_loss"], "b-", label="Train")
axes[0,0].plot(df["epoch"], df["val/box_loss"], "r--", label="Val")
axes[0,0].set_title("Box Loss"); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(df["epoch"], df["train/kobj_loss"], "b-", label="Train")
axes[0,1].plot(df["epoch"], df["val/kobj_loss"], "r--", label="Val")
axes[0,1].set_title("Keypoint Object Loss"); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

axes[0,2].plot(df["epoch"], df["metrics/mAP50"], "g-", linewidth=2, label=f"mAP@0.5 ({map50:.3f})")
axes[0,2].plot(df["epoch"], df["metrics/mAP50-95"], "orange", linewidth=2, linestyle="--", label=f"mAP@0.5:0.95 ({map50_95:.3f})")
axes[0,2].set_title("Box mAP"); axes[0,2].legend(); axes[0,2].set_ylim(0, 1.05)

axes[1,0].plot(df["epoch"], df["metrics/kp_mAP50"], "g-", linewidth=2, label=f"KP mAP@0.5 ({kp_map50:.3f})")
axes[1,0].plot(df["epoch"], df["metrics/kp_mAP50-95"], "orange", linewidth=2, linestyle="--", label=f"KP mAP@0.5:0.95 ({kp_map50_95:.3f})")
axes[1,0].set_title("Keypoint mAP"); axes[1,0].legend(); axes[1,0].set_ylim(0, 1.05)

# FIX: Nếu không có per-keypoint data, hiển thị text thay vì histogram rỗng
ax = axes[1,1]
if kp_ap50_list is not None:
    ax.hist(kp_ap50_list, bins=20, color="#8e44ad", edgecolor="white", alpha=0.8)
    ax.axvline(kp_map50, color="red", linestyle="--", linewidth=2, label=f"Mean = {kp_map50:.3f}")
    ax.set_title(f"Phân bố mAP@0.5 ({n_kp} keypoints)")
    ax.legend()
else:
    ax.axis("off")
    ax.text(0.1, 0.5, f"KP mAP@0.5 = {kp_map50:.4f}\nKP mAP@0.5:0.95 = {kp_map50_95:.4f}\n\n(Không có per-keypoint data\nresults.pose.ap50 là 1 giá trị duy nhất)",
            va="center", fontsize=11, fontweight="bold")
ax.grid(alpha=0.3)

ax = axes[1,2]
ax.axis("off")
if kp_ap50_list is not None:
    visible = [(i, v) for i, v in enumerate(kp_ap50_list) if v > 0]
    top5 = sorted(visible, key=lambda x: -x[1])[:5]
    bot5 = sorted(visible, key=lambda x: x[1])[:5]
    text = "Top 5 KPs:\n" + "\n".join(f"  KP{i}: {v:.3f}" for i, v in top5)
    text += "\n\nBottom 5 KPs:\n" + "\n".join(f"  KP{i}: {v:.3f}" for i, v in bot5)
else:
    text = f"Box mAP@0.5 = {map50:.4f}\nKP mAP@0.5 = {kp_map50:.4f}\nKP mAP@0.5:0.95 = {kp_map50_95:.4f}"
ax.text(0.1, 0.9, text, va="top", fontsize=10, family="monospace",
        bbox=dict(boxstyle="round", facecolor="#f5f5f5"))

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_PNG}")
os.remove(fixed_yaml)